# EAPS 507 — Turning Scientific Equations into a Python Module
## Student Version

### Objective
In this exercise, you will translate five atmospheric-science equations into reusable Python functions and organize them into a Python module.

By the end of the exercise, you should be able to:
- translate mathematical equations into Python,
- write functions with clear inputs and outputs,
- document functions using docstrings,
- place related functions in a `.py` module,
- import your module into a Jupyter notebook, and
- test whether your functions return physically reasonable values.

### Submission
Submit your completed notebook and your Python module file named `atmos_equations.py`.

Before submitting, restart the kernel and run all cells from top to bottom so execution numbers appear in order.

## General Requirements

Create a file named:

```text
atmos_equations.py
```

Your module must contain one function for each of the five equations below. Each function should include a descriptive function name, clearly named arguments, a short docstring, the equation implemented correctly, a returned value, and appropriate units in the docstring.

You may use `numpy` where needed.

# Equation 1 — Potential Temperature

\[
$\theta = T\left(\frac{p_0}{p}\right)^{R_d/c_p}$
\]

where ($\theta$) is potential temperature (K), (T) is temperature (K), (p) is pressure (hPa), ($p_0$=1000\) hPa, \($R_d$=287.05 Jkg$^{-1}$ K$^{-1}$), and ($c_p=1004.0 J kg^{-1} K^{-1}$).


### Task
Write:

```python
potential_temperature(T, p)
```

Test with `T = 288 K` and `p = 850 hPa`.

In [7]:
def potential_temperature(T,p):
    """
    Compute potential temperature for a specific temperature T
    Args:
        T: temperature in Kelvin (K)
        p: pressure in hPa
    Returns:
        theta: potential temperature in Kelvin
    """
    
    p_0 = 1000
    R_d = 287.05
    c_p = 1004
    # Calculate theta with equation
    theta = T * (p_0/p) ** (R_d/ c_p)
    return theta

potential_temperature(288,850)

301.69774688010426

In [8]:
from metpy import calc
from metpy.units import units


calc.potential_temperature(850. * units.mbar, 288. * units.kelvin)


<Quantity(301.688329, 'kelvin')>

In [ ]:
# import atmos_equations as ae
# Test Equation 1 here

# Equation 2 — Saturation Vapor Pressure

Use the Bolton approximation:

\[
$e_s(T_c)=6.112\exp\left(\frac{17.67T_c}{T_c+243.5}\right)$
\]

where \(e_s\) is saturation vapor pressure (hPa) and \($T_c$\) is temperature in °C.

### Task
Write:

```python
saturation_vapor_pressure(Tc)
```

Test with `Tc = 20 °C`.

In [1]:
import atmos_equations as ae
# saturation_vapor_pressure(20)
ae?

Type:        module
String form: <module 'atmos_equations' from '/home/lcrother/EAPS507/Students/atmos_equations.py'>
File:        ~/EAPS507/Students/atmos_equations.py
Docstring:   Contains atmospheric equations

In [12]:
import numpy as np
def saturation_vapor_pressure(Tc):
    """
    Calculate the saturation vapor pressure given temperature
    Args:
        Tc: temperature in C
    Returns:
        e_s: saturation vapor pressure in hPa
    """
    # Calculate saturation vapor pressure with equation and return
    e_s = 6.112 * np.exp(17.67 * Tc / (Tc + 243.5))
    return e_s

saturation_vapor_pressure(20)

np.float64(23.36947123406443)

In [13]:
calc.saturation_vapor_pressure(20 * units.degC, phase='liquid').to('hPa')


<Quantity(23.3474812, 'hectopascal')>

In [ ]:
# Test Equation 2 here

# Equation 3 — Saturation Mixing Ratio

\[
$w_s=\epsilon\frac{e_s}{p-e_s}$
\]

where \($w_s$\) is saturation mixing ratio ($kg kg^{-1}$), ($\epsilon=0.622$), \($e_s$\) is saturation vapor pressure (hPa), and \(p\) is atmospheric pressure (hPa).

### Task
Write:

```python
saturation_mixing_ratio(Tc, p)
```

Your function must call `saturation_vapor_pressure()` rather than rewrite that equation.

Test with `Tc = 20 °C` and `p = 1000 hPa`. Also print the result in g kg\(^{-1}\).

In [19]:
def saturation_mixing_ratio(Tc,p):
    """
    Calculates saturation mixing ratio given temperature and pressure.
    Calls saturation_vapor_pressure() to get the saturation vapor pressure
    Args:
        Tc: temperature in C
        p: pressure in hPa
    Returns:
        w_s: saturation mixing ratio in (gkg^-1)
    """
    e = 0.622
    # Get saturation vapor pressure
    es = saturation_vapor_pressure(Tc)
    # Calculate saturation mixing ratio with equation and return
    ws = e * es / (p - es)
    # Convert to g/kg
    ws *= 1000
    return ws
saturation_mixing_ratio(20,1000)

np.float64(14.883633758567266)

In [20]:
print(calc.saturation_mixing_ratio(1000 * units.hPa, 20 * units.degC).to('g/kg'))

14.868263804532658 gram / kilogram


In [ ]:
# Test Equation 3 here

# Equation 4 — Hypsometric Equation

\[
$z_2-z_1=\frac{R_d\overline{T_v}}{g}\ln\left(\frac{p_1}{p_2}\right)$
\]

where ($z_2-z_1$) is layer thickness (m), ($R_d=287.05 J kg^{-1} K^{-1}$), ($\overline{T_v}$) is layer-mean virtual temperature (K), ($g=9.80665 ms^{-2})$, ($p_1$) is lower-level pressure, and ($p_2$) is upper-level pressure.

### Task
Write:

```python
hypsometric_thickness(Tv, p1, p2)
```

Test for the 1000–500 hPa layer using `Tv = 270 K`, `p1 = 1000 hPa`, and `p2 = 500 hPa`.

In [21]:
def hypsometric_thickness(Tv, p1, p2):
    """
    Calculates layer thickness given layer-mean virtual temperature, lower-level pressure, an upper-level pressure.
    Args:
        Tv: layer-mean virtual temperature in K
        p1: lower-level pressure in hPa
        p2: upper-level pressure in hPa
    Returns:
        z1_z2: layer thickness in meters
    """
    Rd = 287.05
    g = 9.80665
    # Calculate layer thickness and return
    z2_z1 = (Rd * Tv / g) * np.log(p1 / p2)
    return z2_z1
hypsometric_thickness(270, 1000, 500)

np.float64(5478.051374172395)

In [28]:
temp = [270,270] * units.K
p1 = [1000,500] * units.hPa
calc.thickness_hydrostatic(pressure=p1, temperature=temp)

<Quantity(5478.00349, 'meter')>

In [ ]:
# Test Equation 4 here

# Equation 5 — Geostrophic Wind

\[
$u_g=-\frac{g}{f}\frac{\partial Z}{\partial y}$
\]

\[
$v_g=\frac{g}{f}\frac{\partial Z}{\partial x}$
\]

with

\[
$f=2\Omega\sin\phi$
\]

where ($u_g$) and ($v_g$) are geostrophic wind components ($ms^{-1}$), ($g=9.80665 ms^{-2}$), ($Z$) is geopotential height (m), ($f$) is the Coriolis parameter ($s^{-1}$), ($\Omega=7.2921\times10^{-5} s^{-1}$), and ($\phi$) is latitude.

### Task
Write:

```python
geostrophic_wind(dZdx, dZdy, latitude)
```

The function must return both `ug` and `vg`.

Test with `dZdx = 2.0e-5`, `dZdy = -3.0e-5`, and `latitude = 40°`.

In [29]:
def geostrophic_wind(dZdx, dZdy, latitude):
    """
    Calculates geostrophic wind components given rate of change of latitude with respect to x and y and current latitude
    Args:
        dZdx: rate of change of height with respect to x
        dZdy: rate of change of height with respect to y
        latitude: latitude
    Returns:
        ug, vg: geostrophic wind components
    """
    omega = 7.2921
    g = 9.80665
    # Compute Coriolis parameter
    f = 2 * omega * np.sin(latitude)
    # Compute geostrophic wind components with the equations
    ug = - (g/f) * dZdy
    vg = (g/f) * dZdx
    return ug,vg
geostrophic_wind(2.0e-5, -3.0e-5, 40)

(np.float64(2.7073043731081026e-05), np.float64(1.8048695820720684e-05))

In [40]:
calc.geostrophic_wind([1,2,2] * units.m, [2.0e-5] * units("m"), [-3.0e-5] * units("m"), 40 * units.degrees_north)

AxisError: axis -2 is out of bounds for array of dimension 1

In [ ]:
# Test Equation 5 here

# Final Integration Test

Once your module is complete, import all five functions and demonstrate that they work in one cell.

Your final cell should calculate and print:
1. potential temperature,
2. saturation vapor pressure,
3. saturation mixing ratio,
4. 1000–500 hPa thickness,
5. both geostrophic wind components.

Include units in every printed result.

In [ ]:
# import atmos_equations as ae
# Write your final integration test here

# Reflection

In **4–6 sentences**, answer:
1. Why is placing equations in a module better than repeatedly rewriting them in a notebook?
2. Which equation was most difficult to translate into Python, and why?
3. What kinds of errors can occur when scientific units are not handled carefully?